# 工程设变价格跟踪

工程设变后：EBOM 零件版本升级 → EPS 接收价格变化 → MRP 新版本订单收货 → 检查价格维护状态。


In [ ]:
import pandas as pd
from datetime import datetime
import os

START_DATE = "2026-06-15"
END_DATE = "2026-06-20"
TODAY = datetime.now().strftime("%Y%m%d")
OUTPUT_DIR = "../output/工程设变价格跟踪"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, f"remind_{TODAY}.xlsx")
print(f"运行日期: {TODAY}  查询区间: {START_DATE} ~ {END_DATE}")

## Step 1：读取 EBOM 设变版本表

In [ ]:
df_ebom = pd.read_excel("../data/EBOM_设变版本表.xlsx")
df_ebom["设变生效日期"] = pd.to_datetime(df_ebom["设变生效日期"], errors="coerce")
df_ebom_latest = df_ebom.sort_values(["零件号","设变生效日期"], ascending=[True,False]).groupby("零件号", as_index=False).first()
print(f"EBOM: {len(df_ebom)} rows -> {len(df_ebom_latest)} unique parts")

## Step 2：读取 MRP 收货记录

In [ ]:
df_mrp = pd.read_excel("../data/MRP_收货记录表.xlsx")
df_mrp["收货日期"] = pd.to_datetime(df_mrp["收货日期"], errors="coerce")
mask = (df_mrp["收货日期"] >= START_DATE) & (df_mrp["收货日期"] <= END_DATE)
df_mrp_filtered = df_mrp[mask].copy()
print(f"MRP filtered: {len(df_mrp_filtered)} rows")

## Step 3：读取 EPS 价格数据

In [ ]:
df_eps = pd.read_excel("../data/eps3_mass.xlsx")
df_eps_valid = df_eps[df_eps["状态"].isin(["已发布","已确认","已审批"])].copy()
df_eps_valid["mk"] = df_eps_valid["零件号"].astype(str)+'_'+df_eps_valid["供应商编码"].astype(str)
df_eps_valid["创建时间"] = pd.to_datetime(df_eps_valid["创建时间"], errors="coerce")
df_eps_latest = df_eps_valid.sort_values(["mk","创建时间"], ascending=[True,False]).groupby("mk", as_index=False).first()
print(f"EPS valid: {len(df_eps_latest)}")

## Step 4：三源交叉匹配

In [ ]:
df_mrp_filtered["mk"] = df_mrp_filtered["零件号"].astype(str)+'_'+df_mrp_filtered["供应商编码"].astype(str)
df_ebom_latest["mk"] = df_ebom_latest["零件号"].astype(str)+'_'+df_ebom_latest["供应商编码"].astype(str)

# MRP LEFT JOIN EBOM
df_m = df_mrp_filtered.merge(df_ebom_latest[["mk","零件名称","新版本号","工程设变号","设变说明","采购工程师"]], on="mk", how="left")
with_eco = df_m[df_m["工程设变号"].notna()]
print(f"MRP {len(df_m)} -> has ECO: {len(with_eco)}")

# LEFT JOIN EPS
df_r = with_eco.merge(df_eps_latest[["mk","零件裸价","价格通知书号"]], on="mk", how="left", suffixes=("","_eps"))
df_remind = df_r[df_r["价格通知书号"].isna()].copy()
print(f"待维护: {len(df_remind)}")
if len(df_remind)>0: print(df_remind.groupby("采购工程师").size().to_string())

## Step 5：导出

In [ ]:
if len(df_remind)>0:
    cols=["零件号","零件名称","新版本号","工程设变号","设变说明","订单号","收货日期","收货数量","版本号","供应商编码","供应商名称","采购工程师"]
    df_out=df_remind[cols].copy()
    df_out.rename(columns={"版本号":"订单中版本号"},inplace=True)
    df_out["检查日期"]=datetime.now().strftime("%Y/%m/%d")
    os.makedirs(OUTPUT_DIR,exist_ok=True)
    df_out.to_excel(OUTPUT_FILE,index=False)
    print(f"Exported: {OUTPUT_FILE} ({len(df_out)} rows)")
else:
    print("无待维护项")